# 🚀 HUẤN LUYỆN YOLO26x - MÔ HÌNH THỊ GIÁC MỚI NHẤT HIỆN NAY (SOTA 2026)
### Nhận diện biển số xe Việt Nam toàn diện trên Google Cloud (GPU NVIDIA L4/T4) & Kaggle
**Dự án:** `yolov26-nhandien-biensoxe`  

### 🌟 Toàn bộ công nghệ mới nhất được tích hợp:
1. **Mô hình đầu bảng:** `YOLO26x` (Extra Large - bản lớn nhất, độ chính xác cao nhất của Ultralytics 2026).
2. **Native NMS-Free End-to-End:** Xuất kết quả trực tiếp 1-to-1 từ mạng nơ-ron, triệt tiêu hoàn toàn bước lọc NMS truyền thống, tối ưu độ trễ cho camera bãi xe.
3. **Trình tối ưu hóa MuSGD Optimizer:** Kết hợp Momentum SGD với Muon cho tốc độ hội tụ nhanh vượt trội.
4. **Tích hợp ONNXSLIM:** Công nghệ tối ưu đồ thị ONNX mới nhất, thay thế hoàn toàn onnx-simplifier cũ.
5. **Hợp nhất kho dữ liệu biển số Việt Nam khổng lồ:**
   - `tanhphp/vietnamese-license-plates` (1.1 GB - Bộ dữ liệu cập nhật mới nhất 2026).
   - `bomaich/vnlicenseplate` (Bộ dữ liệu biển xe máy & ô tô thực tế).
   - `phatphat9blht/vietnamese-license-plate-yolov26` (Roboflow VN Plate).
   - Đầy đủ biển xe máy 2 dòng vuông, ô tô 1 dòng dài, xe tải, xe điện, góc camera xiên, ban đêm đèn pha chói.

## Bước 1: Cài đặt Ultralytics & ONNXSLIM mới nhất

In [ ]:
!pip install -q --upgrade ultralytics onnx onnxruntime onnxslim albumentations kaggle pyyaml
import os, shutil, glob
from pathlib import Path
import torch
import ultralytics

print(f"Ultralytics Version: {ultralytics.__version__}")
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
    print(f"GPU VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")


## Bước 2: Tự động tải 3 bộ dữ liệu biển số xe Việt Nam lớn nhất

In [ ]:
# Cấu hình Kaggle API Token của bạn để tự động tải
os.environ["KAGGLE_API_TOKEN"] = "KGAT_3afa26c419f771f34ee8dbb8003501bc"

DATA_DIR = Path("/content/dataset") if os.path.exists("/content") else Path("./dataset")
DATA_DIR.mkdir(parents=True, exist_ok=True)

print("1/3. Đang tải bộ dữ liệu tanhphp (1.1 GB, 2026 mới nhất)...")
!kaggle datasets download -d tanhphp/vietnamese-license-plates -p {DATA_DIR}/raw_tanh --unzip -q

print("2/3. Đang tải bộ dữ liệu bomaich (VNLicensePlate xe máy & ô tô)...")
!kaggle datasets download -d bomaich/vnlicenseplate -p {DATA_DIR}/raw_vn --unzip -q

print("3/3. Đang tải bộ dữ liệu Roboflow VN Plate...")
!kaggle datasets download -d phatphat9blht/vietnamese-license-plate-yolov26 -p {DATA_DIR}/raw_rf --unzip -q

print("TẢI THÀNH CÔNG TOÀN BỘ 3 TẬP DỮ LIỆU!")


## Bước 3: Hợp nhất toàn bộ dữ liệu & Chuẩn hóa định dạng nhãn
Tập trung nhận diện biển số xe (Class 0) kết hợp các lớp người & cửa bãi xe.

In [ ]:
import yaml

TRAIN_IMG = DATA_DIR / "images/train"
VAL_IMG = DATA_DIR / "images/val"
TRAIN_LBL = DATA_DIR / "labels/train"
VAL_LBL = DATA_DIR / "labels/val"

for p in [TRAIN_IMG, VAL_IMG, TRAIN_LBL, VAL_LBL]:
    p.mkdir(parents=True, exist_ok=True)

def copy_dataset_pairs(img_folder, lbl_folder, target_img, target_lbl, prefix):
    if not img_folder.exists(): return
    for img_file in img_folder.glob("*.*"):
        if img_file.suffix.lower() in [".jpg", ".jpeg", ".png"]:
            lbl_file = lbl_folder / f"{img_file.stem}.txt"
            if lbl_file.exists():
                shutil.copy2(img_file, target_img / f"{prefix}_{img_file.name}")
                shutil.copy2(lbl_file, target_lbl / f"{prefix}_{lbl_file.name}")

# 1. Hợp nhất tanhphp (1.1 GB)
copy_dataset_pairs(DATA_DIR / "raw_tanh/splits/train/images", DATA_DIR / "raw_tanh/splits/train/labels", TRAIN_IMG, TRAIN_LBL, "tanh")
copy_dataset_pairs(DATA_DIR / "raw_tanh/splits/test/images", DATA_DIR / "raw_tanh/splits/test/labels", VAL_IMG, VAL_LBL, "tanh")

# 2. Hợp nhất bomaich (VNLicensePlate)
copy_dataset_pairs(DATA_DIR / "raw_vn/train/images", DATA_DIR / "raw_vn/train/labels", TRAIN_IMG, TRAIN_LBL, "vn")
copy_dataset_pairs(DATA_DIR / "raw_vn/test/images", DATA_DIR / "raw_vn/test/labels", VAL_IMG, VAL_LBL, "vn")

# 3. Hợp nhất Roboflow
copy_dataset_pairs(DATA_DIR / "raw_rf/train/images", DATA_DIR / "raw_rf/train/labels", TRAIN_IMG, TRAIN_LBL, "rf")
copy_dataset_pairs(DATA_DIR / "raw_rf/valid/images", DATA_DIR / "raw_rf/valid/labels", VAL_IMG, VAL_LBL, "rf")

train_count = len(list(TRAIN_IMG.glob("*.*")))
val_count = len(list(VAL_IMG.glob("*.*")))
print(f"\n=== TỔNG HỢP DỮ LIỆU HOÀN TẤT ===")
print(f"- Số ảnh Train (Huấn luyện): {train_count} ảnh")
print(f"- Số ảnh Val (Kiểm thử):     {val_count} ảnh")

# Tạo file data.yaml chuẩn 100% với hệ thống backend
data_yaml = {
    "path": str(DATA_DIR.resolve()),
    "train": "images/train",
    "val": "images/val",
    "names": {
        0: "license_plate",
        1: "face",
        2: "door_open",
        3: "door_closed"
    }
}

yaml_path = DATA_DIR / "data.yaml"
with open(yaml_path, "w", encoding="utf-8") as f:
    yaml.dump(data_yaml, f, sort_keys=False)

print(f"File cấu hình dữ liệu sẵn sàng: {yaml_path}")


## Bước 4: Huấn luyện YOLO26x với siêu tham số SOTA bãi xe / barrier

In [ ]:
from ultralytics import YOLO

# Nạp model YOLO26x (Bản Extra Large mạnh nhất của thế hệ YOLO26)
model = YOLO("yolo26x.pt")

results = model.train(
    data=str(yaml_path),
    epochs=100,
    imgsz=640,
    batch=32 if torch.cuda.is_available() else 8,
    device=0 if torch.cuda.is_available() else "cpu",
    optimizer="auto",       # Tự động chọn MuSGD trên YOLO26
    cos_lr=True,          # Cosine learning rate schedule giúp hội tụ mượt mà
    close_mosaic=10,      # Tắt mosaic ở 10 epoch cuối để viền box cực kỳ chuẩn nét
    # Siêu tham số Data Augmentation thực chiến:
    mosaic=1.0,           # Ghép 4 ảnh học bối cảnh xe đa dạng
    mixup=0.20,           # Tăng khả năng chịu biển bị che khuất 1 phần
    copy_paste=0.20,      # Dán biển số vào các khung cảnh khác nhau
    degrees=15.0,         # Chống xoay nghiêng biển số
    perspective=0.001,    # Chống góc camera nhìn chéo từ trên cột cao xuống
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,            # Chống lóa đèn pha ô tô/xe máy ban đêm & thiếu sáng
    scale=0.5,            # Học nhận diện cả biển siêu nhỏ ở xa lẫn biển to gần
    save=True,
    plots=True,
    project="runs_yolo26",
    name="vn_plate_sota_yolo26x"
)


## Bước 5: Đánh giá độ chính xác (mAP) của mô hình

In [ ]:
best_weights = Path("runs_yolo26/vn_plate_sota_yolo26x/weights/best.pt")
val_model = YOLO(str(best_weights))
metrics = val_model.val()
print(f"\n==================== KẾT QUẢ ĐÁNH GIÁ YOLO26x ====================")
print(f"mAP@50:    {metrics.box.map50:.4f}")
print(f"mAP@50-95: {metrics.box.map:.4f}")
print(f"Precision: {metrics.box.mp:.4f}")
print(f"Recall:    {metrics.box.mr:.4f}")


## Bước 6: Xuất ONNX (FP32 & INT8 Quantization) tối ưu hóa bằng ONNXSLIM

In [ ]:
# 6.1 Xuất ONNX FP32 (YOLO26 Native End-to-End NMS-Free)
onnx_file = val_model.export(
    format="onnx",
    imgsz=640,
    dynamic=False,
    simplify=True,        # Tự động tối ưu bằng ONNXSLIM
    opset=17,             # Opset mới nhất của ONNX
    nms=False
)
print("Xuất ONNX hoàn tất:", onnx_file)

export_dir = Path("exported_models")
export_dir.mkdir(exist_ok=True)
target_fp32 = export_dir / "yolo26_plate_people_door.onnx"
shutil.copy2(onnx_file, target_fp32)

# 6.2 Lượng tử hóa INT8 (Dynamic Quantization) - Tăng tốc độ gấp 2-3 lần trên CPU máy trạm
import onnxruntime
from onnxruntime.quantization import quantize_dynamic, QuantType

target_int8 = export_dir / "yolo26_plate_people_door_int8.onnx"
quantize_dynamic(
    model_input=str(target_fp32),
    model_output=str(target_int8),
    weight_type=QuantType.QUInt8
)
print("Xuất INT8 hoàn tất:", target_int8)

# Đóng gói zip tải về
!zip -j model_yolo26x_vietnamese_plate.zip exported_models/*
print("\n=================== HOÀN TẤT TOÀN BỘ ===================")
print("Hãy tải file 'model_yolo26x_vietnamese_plate.zip' về máy và chép vào thư mục: backend/models/")
